In [0]:
catalogo_colunas = spark.sql("""
    SELECT
        table_name AS tabela,
        ordinal_position AS posicao,
        column_name AS coluna,
        full_data_type AS tipo,
        comment AS descricao
    FROM workspace.information_schema.columns
    WHERE table_schema = 'mvp_gramados'
    ORDER BY table_name, ordinal_position
""")

display(catalogo_colunas)

In [0]:
from pyspark.sql import functions as F

display(
    catalogo_colunas
    .groupBy("tabela")
    .agg(
        F.count("*").alias("total_colunas"),
        F.sum(
            F.when(
                F.col("descricao").isNull()
                | (F.trim(F.col("descricao")) == ""),
                1
            ).otherwise(0)
        ).alias("colunas_sem_descricao")
    )
    .orderBy("tabela")
)

In [0]:
descricoes_partidas = {
    "ID": "Identificador da partida na fonte. Esperado: inteiro positivo e unico, armazenado como texto na Bronze.",
    "rodata": "Numero da rodada, armazenado como texto. Nome original da coluna preservado da fonte. Esperado: inteiro positivo.",
    "data": "Data da partida na fonte, em texto no formato dd/MM/yyyy.",
    "hora": "Horario da partida informado pela fonte, armazenado como texto.",
    "mandante": "Nome do clube mandante conforme registrado na fonte.",
    "visitante": "Nome do clube visitante conforme registrado na fonte.",
    "formacao_mandante": "Formacao tatica do mandante informada pela fonte. Pode estar ausente.",
    "formacao_visitante": "Formacao tatica do visitante informada pela fonte. Pode estar ausente.",
    "tecnico_mandante": "Nome do tecnico do mandante informado pela fonte. Pode estar ausente.",
    "tecnico_visitante": "Nome do tecnico do visitante informado pela fonte. Pode estar ausente.",
    "vencedor": "Vencedor informado pela fonte; hifen representa empate. Campo original preservado, sujeito a divergencias com o placar.",
    "arena": "Nome do estadio conforme registrado na fonte. Um estadio pode possuir diferentes nomes.",
    "mandante_Placar": "Gols do mandante, armazenados como texto. Esperado: inteiro maior ou igual a zero.",
    "visitante_Placar": "Gols do visitante, armazenados como texto. Esperado: inteiro maior ou igual a zero.",
    "mandante_Estado": "Sigla da unidade federativa do clube mandante, conforme a fonte.",
    "visitante_Estado": "Sigla da unidade federativa do clube visitante, conforme a fonte."
}

descricoes_gramados = {
    "arena": "Nome do estadio usado para associar a pesquisa as partidas por igualdade de texto.",
    "tipo_gramado": "Classificacao historica pesquisada: natural, sintetico, hibrido ou nao_confirmado.",
    "inicio_validade": "Inicio inclusivo do intervalo adotado na pesquisa para classificar os jogos. Nao representa necessariamente a data de instalacao do gramado.",
    "fim_validade": "Fim inclusivo do intervalo adotado na pesquisa para classificar os jogos. Nao representa necessariamente a data de retirada do gramado.",
    "fonte_url": "URL ou URLs das referencias consultadas. O preenchimento nao garante que a classificacao esteja confirmada.",
    "observacoes": "Notas da pesquisa, incluindo nomes alternativos, evidencias, inferencias e pendencias."
}

descricoes_gold = {
    "tipo_gramado": "Tipo de gramado das partidas elegiveis: natural, sintetico ou hibrido.",
    "quantidade_partidas": "Quantidade de partidas do grupo com status dentro_do_periodo. Inteiro positivo.",
    "media_gols": "Media da soma dos gols do mandante e do visitante nas partidas do grupo. Valor maior ou igual a zero.",
    "vitorias_mandante": "Quantidade de partidas do grupo em que o placar do mandante supera o do visitante. Entre zero e quantidade_partidas.",
    "taxa_vitoria_mandante": "Proporcao de vitorias do mandante no grupo: vitorias_mandante divididas por quantidade_partidas. Valor entre 0 e 1."
}

metadados = {
    "bronze_partidas": descricoes_partidas,
    "bronze_gramados": {
        coluna: descricao + " Origem: gramados.csv; valor carregado como texto."
        for coluna, descricao in descricoes_gramados.items()
    },
    "silver_gramados": {
        coluna: descricao + (
            " Origem: bronze_gramados; convertido para DATE, com nulo quando ausente ou nao conversivel."
            if coluna in ("inicio_validade", "fim_validade")
            else " Origem: bronze_gramados."
        )
        for coluna, descricao in descricoes_gramados.items()
    },
    "gold_resumo_gramado": descricoes_gold,
    "gold_resumo_ano_gramado": {
        **descricoes_gold,
        "ano": "Ano extraido da data da partida. Valores do recorte: 2023 e 2024."
    },
    "gold_resumo_clube_gramado": {
        **descricoes_gold,
        "mandante": "Clube mandante usado no agrupamento junto ao tipo de gramado."
    }
}

# Confere os nomes antes de aplicar os comentarios.
for tabela, colunas in metadados.items():
    existentes = set(
        spark.table(f"workspace.mvp_gramados.{tabela}").columns
    )
    faltantes = set(colunas) - existentes
    if faltantes:
        raise ValueError(f"Colunas nao encontradas em {tabela}: {faltantes}")

quantidade = 0

for tabela, colunas in metadados.items():
    for coluna, descricao in colunas.items():
        texto_sql = descricao.replace("'", "''")
        spark.sql(
            f"ALTER TABLE workspace.mvp_gramados.`{tabela}` "
            f"ALTER COLUMN `{coluna}` COMMENT '{texto_sql}'"
        )
        quantidade += 1

print(f"Descricoes aplicadas: {quantidade}")

In [0]:
display(spark.sql("""
    SELECT
        table_name AS tabela,
        COUNT(*) AS total_colunas,
        SUM(
            CASE
                WHEN comment IS NULL OR TRIM(comment) = '' THEN 1
                ELSE 0
            END
        ) AS colunas_sem_descricao
    FROM workspace.information_schema.columns
    WHERE table_schema = 'mvp_gramados'
    GROUP BY table_name
    ORDER BY table_name
"""))